Freight-Rate-Prediction

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

DATA = "/content/drive/MyDrive/freight-rate-ml/"
df = pd.read_csv(DATA + "train_test.csv", parse_dates=["date"])
df.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [3]:
print(df.shape)
df.info()

(48000, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   load_id       48000 non-null  object        
 1   pickup        48000 non-null  object        
 2   delivery      48000 non-null  object        
 3   pickup_lat    48000 non-null  float64       
 4   pickup_lon    48000 non-null  float64       
 5   delivery_lat  48000 non-null  float64       
 6   delivery_lon  48000 non-null  float64       
 7   distance      48000 non-null  float64       
 8   equipment     48000 non-null  object        
 9   weight        47700 non-null  float64       
 10  date          48000 non-null  datetime64[ns]
 11  market_index  47626 non-null  float64       
 12  quote_signal  48000 non-null  float64       
 13  posted_rate   48000 non-null  float64       
dtypes: datetime64[ns](1), float64(9), object(4)
memory usage: 5.1+ MB


In [4]:
df.isna().sum()

,0
load_id,0
pickup,0
delivery,0
pickup_lat,0
pickup_lon,0
delivery_lat,0
delivery_lon,0
distance,0
equipment,0
weight,300


In [5]:
df.describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,date,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,48000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,2025-05-31 19:53:29.400000,1.083387,2.062468,2373.980682
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,2025-01-01 00:00:00,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,2025-03-18 00:00:00,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,2025-05-31 00:00:00,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,2025-08-15 00:00:00,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,2025-10-31 00:00:00,1.467780,3.610350,25533.000000
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,NaN,0.168091,0.291391,1486.493245


In [6]:
print("Negative weight:", (df["weight"] < 0).sum())
print("Zero/negative distance:", (df["distance"] <= 0).sum())
print("Same pickup & delivery:", (df["pickup"] == df["delivery"]).sum())
print("Duplicate load_id:", df["load_id"].duplicated().sum())

Negative weight: 292
Zero/negative distance: 0
Same pickup & delivery: 0
Duplicate load_id: 0


In [7]:
df["rpm"] = df["posted_rate"] / df["distance"]
df["rpm"].describe(percentiles=[0.01, 0.5, 0.99])

,rpm
count,48000.000000
mean,2.215347
std,0.583887
min,0.333709
1%,1.673521
50%,2.145348
99%,3.176940
max,14.125294


In [8]:
print(df.groupby("equipment")["rpm"].median(), "\n")
print(df.groupby(df["date"].dt.month)["rpm"].median().round(3), "\n")
print(df.groupby(df["date"].dt.dayofweek)["rpm"].median().round(3))

equipment
Dry Van    2.046203
Flatbed    2.219683
Reefer     2.311540
Name: rpm, dtype: float64 

date
1     2.029
2     2.061
3     2.139
4     2.147
5     2.190
6     2.248
7     2.187
8     2.121
9     2.160
10    2.164
Name: rpm, dtype: float64 

date
0    2.126
1    2.139
2    2.163
3    2.171
4    2.157
5    2.132
6    2.127
Name: rpm, dtype: float64


In [9]:
df[["posted_rate", "distance", "weight", "market_index", "quote_signal"]].corr()["posted_rate"]

,posted_rate
posted_rate,1.000000
distance,0.908519
weight,0.034840
market_index,0.034165
quote_signal,-0.039858


In [10]:
val = pd.read_csv(DATA + "validation.csv", parse_dates=["date"])
print("Train:", df["date"].min().date(), "to", df["date"].max().date())
print("Validation:", val["date"].min().date(), "to", val["date"].max().date())

train_cities = set(df["pickup"]) | set(df["delivery"])
new_cities = (set(val["pickup"]) | set(val["delivery"])) - train_cities
print("Cities never seen in training:", new_cities)

Train: 2025-01-01 to 2025-10-31
Validation: 2025-11-01 to 2025-12-31
Cities never seen in training: {'San Diego', 'Chicago', 'Jackson', 'Allentown', 'Charlotte', 'Norfolk', 'Laredo', 'Knoxville'}


In [11]:
raw_df = df.copy()                      # backup of original data

df["weight"] = df["weight"].abs()       # negative -> positive

before = len(df)
df = df[(df["rpm"] > 1.2) & (df["rpm"] < 4)].copy()
print("Removed outliers:", before - len(df))

Removed outliers: 675


In [12]:
FEATURES = ["pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon",
            "distance", "weight", "equipment", "quote_signal", "day_of_week"]

def make_features(data):
    X = data.copy()
    X["day_of_week"] = X["date"].dt.dayofweek
    X["equipment"] = pd.Categorical(X["equipment"],
                                    categories=["Dry Van", "Flatbed", "Reefer"])
    return X[FEATURES]

In [13]:
from sklearn.metrics import mean_absolute_error

train = df[df["date"] < "2025-09-01"]     # Jan–Aug
test  = df[df["date"] >= "2025-09-01"]    # Sep–Oct
print("Train:", len(train), "| Test:", len(test))

median_rpm = train.groupby("equipment")["rpm"].median()
base_pred = test["distance"] * test["equipment"].map(median_rpm)
print("Baseline MAE:", round(mean_absolute_error(test["posted_rate"], base_pred), 2))

Train: 37946 | Test: 9379
Baseline MAE: 175.08


In [14]:
import lightgbm as lgb

model = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03,
                          objective="l1", random_state=42, verbose=-1)
model.fit(make_features(train), train["rpm"])

pred = model.predict(make_features(test)) * test["distance"]
print("LightGBM MAE:", round(mean_absolute_error(test["posted_rate"], pred), 2))
print("MAPE %:", round((abs(test["posted_rate"] - pred) / test["posted_rate"]).mean() * 100, 2))

pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

LightGBM MAE: 30.04
MAPE %: 1.2


,0
quote_signal,6812
distance,4654
weight,3788
equipment,1701
pickup_lat,1590
delivery_lat,1584
pickup_lon,1412
delivery_lon,1410
day_of_week,1049


In [15]:
def make_features_mi(data):
    X = make_features(data)
    X["market_index"] = data["market_index"]
    return X

m2 = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, objective="l1",
                       random_state=42, verbose=-1)
m2.fit(make_features_mi(train), train["rpm"])
pred2 = m2.predict(make_features_mi(test)) * test["distance"]
print("With market_index MAE:", round(mean_absolute_error(test["posted_rate"], pred2), 2))

df.groupby(df["date"].dt.month)["market_index"].mean().round(3)

With market_index MAE: 76.0


,market_index
date,
1,0.927
2,0.996
3,1.073
4,1.207
5,1.301
6,1.280
7,1.201
8,0.981
9,0.893


In [16]:
raw_test = raw_df[raw_df["date"] >= "2025-09-01"].copy()
raw_test["weight"] = raw_test["weight"].abs()
raw_pred = model.predict(make_features(raw_test)) * raw_test["distance"]
print("MAE incl. outliers:", round(mean_absolute_error(raw_test["posted_rate"], raw_pred), 2))

MAE incl. outliers: 86.15


In [17]:
import os
OUT = DATA + "outputs/"
os.makedirs(OUT, exist_ok=True)

final_model = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, objective="l1",
                                random_state=42, verbose=-1)
final_model.fit(make_features(df), df["rpm"])

val["weight"] = val["weight"].abs()
val["predicted_rate"] = final_model.predict(make_features(val)) * val["distance"]

template = pd.read_csv(DATA + "validation_predictions_template.csv")
out = template[["load_id"]].merge(val[["load_id", "predicted_rate"]], on="load_id", how="left")
out["predicted_rate"] = out["predicted_rate"].round(2)
print(out.shape, "| missing:", out["predicted_rate"].isna().sum())
out.to_csv(OUT + "validation_predictions.csv", index=False)

(12000, 2) | missing: 0


In [18]:
dec = pd.read_csv(DATA + "december_chart_inputs.csv", parse_dates=["date"])

coords = pd.concat([
    df[["pickup", "pickup_lat", "pickup_lon"]].set_axis(["city", "lat", "lon"], axis=1),
    df[["delivery", "delivery_lat", "delivery_lon"]].set_axis(["city", "lat", "lon"], axis=1),
]).drop_duplicates("city").set_index("city")

dec_in = dec.copy()
dec_in["pickup_lat"]   = dec_in["pickup"].map(coords["lat"])
dec_in["pickup_lon"]   = dec_in["pickup"].map(coords["lon"])
dec_in["delivery_lat"] = dec_in["delivery"].map(coords["lat"])
dec_in["delivery_lon"] = dec_in["delivery"].map(coords["lon"])
dec_in["quote_signal"] = df["quote_signal"].median()

dec["predicted_rate"] = (final_model.predict(make_features(dec_in)) * dec_in["distance"]).round(2)
dec["date"] = dec["date"].dt.strftime("%Y-%m-%d")
dec.to_csv(OUT + "december_chart_inputs.csv", index=False)
dec[["date", "predicted_rate"]].head(10)

,date,predicted_rate
0,2025-12-01,773.64
1,2025-12-02,774.35
2,2025-12-03,777.37
3,2025-12-04,778.57
4,2025-12-05,776.80
5,2025-12-06,774.18
6,2025-12-07,773.87
7,2025-12-08,773.64
8,2025-12-09,774.35
9,2025-12-10,777.37


In [20]:
%cd /content/drive/MyDrive/freight-rate-ml
!python score.py --predictions outputs/validation_predictions.csv --december-predictions outputs/december_chart_inputs.csv --output-dir outputs/scorer_results

/content/drive/MyDrive/freight-rate-ml
Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: outputs/scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.
